# Module 9: Factor Analysis - Complete Guide

## 1. Introduction: The Philosophy of Factor Analysis
You have successfully navigated the landscapes of regression and time series. While those methods are primarily concerned with predictive relationships (how X influences Y over time or space), Factor Analysis shifts our focus toward **data reduction and latent structure detection**.

It is common in pharmaceutical and business analytics to face high-dimensional data datasets where you have hundreds of variables (e.g., patient health metrics, regional market KPIs, or survey items). Factor analysis is the primary statistical tool for identifying the hidden signals within this noise.

### Why Factor Analysis?
- **Data Simplification:** Instead of analyzing 50 different KPIs, you might find that 45 of them are heavily correlated and can be represented by 3 super-variables or Factors.
- **Latent Structure:** It allows you to move from observable behavior to underlying characteristics. In pharma, this might mean grouping various symptoms into a single Health Index factor or consolidating complex market variables into Market Sentiment.

In [ ]:
# Always start with imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import FactorAnalysis
from scipy import stats

# Set display options for clarity
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully. Ready for Factor Analysis!")

## 2. Data Creation: High-Dimensional Pharmaceutical KPIs
To understand Factor Analysis, we must construct a dataset where variables are not entirely independent. We will simulate 9 observable business metrics for a pharmaceutical company across 500 regions.

These 9 variables are actually manifestations of 3 unobservable Latent Factors:
1. **Patient Health Profile**
2. **Regional Market Sentiment**
3. **Product Accessibility**

In [ ]:
n_samples = 500

# Step 1: Generate the 3 True Latent Factors (Hidden from the observer)
health_factor = np.random.normal(0, 1, n_samples)
market_factor = np.random.normal(0, 1, n_samples)
access_factor = np.random.normal(0, 1, n_samples)

# Step 2: Generate 9 Observable Variables based on the Latent Factors + Unique Noise
# Group 1: Driven by Health Factor
blood_pressure_issues = 0.8 * health_factor + np.random.normal(0, 0.4, n_samples)
cholesterol_levels = 0.75 * health_factor + np.random.normal(0, 0.5, n_samples)
bmi_index = 0.85 * health_factor + np.random.normal(0, 0.3, n_samples)

# Group 2: Driven by Market Sentiment Factor
sales_growth = 0.8 * market_factor + np.random.normal(0, 0.4, n_samples)
brand_trust_score = 0.9 * market_factor + np.random.normal(0, 0.2, n_samples)
market_share = 0.7 * market_factor + np.random.normal(0, 0.6, n_samples)

# Group 3: Driven by Accessibility Factor
clinic_density = 0.85 * access_factor + np.random.normal(0, 0.3, n_samples)
supply_efficiency = 0.75 * access_factor + np.random.normal(0, 0.5, n_samples)
delivery_speed = 0.8 * access_factor + np.random.normal(0, 0.4, n_samples)

# Step 3: Combine into a DataFrame
df_pharma = pd.DataFrame({
    'BP_Issues': blood_pressure_issues,
    'Cholesterol': cholesterol_levels,
    'BMI': bmi_index,
    'Sales_Growth': sales_growth,
    'Brand_Trust': brand_trust_score,
    'Market_Share': market_share,
    'Clinic_Density': clinic_density,
    'Supply_Efficiency': supply_efficiency,
    'Delivery_Speed': delivery_speed
})

print("Dataset created with 500 regions and 9 observable KPIs.")
print(df_pharma.head())

## 3. Initial Exploration: The Correlation Matrix
Before applying Factor Analysis, we must ensure there are inter-dependencies among our variables. If all variables are perfectly independent (correlation near 0), Factor Analysis will fail because there is no shared variance to group together.

Let's visualize the relationships using a correlation heatmap.

In [ ]:
plt.figure(figsize=(10, 8))
corr_matrix = df_pharma.corr()

# Use a mask to plot only the lower triangle for cleaner visualization
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)

plt.title("Correlation Matrix of Observable KPIs", fontsize=16)
plt.tight_layout()
plt.show()

print("Observation: You can visually detect 3 distinct 'blocks' of high correlation.")
print("This is the exact pattern Factor Analysis looks for to identify latent structures.")

## 4. Assumptions Checking: Bartlett's Test of Sphericity
To mathematically validate what our eyes see in the correlation matrix, we use Bartlett's Test of Sphericity.

**Null Hypothesis:** The correlation matrix is an identity matrix (variables are completely unrelated).
**Alternative Hypothesis:** Variables are related enough to conduct factor analysis.

If the p-value is less than 0.05, we reject the null hypothesis and proceed.

In [ ]:
# Implementing a manual Bartlett's Test using scipy to avoid relying on external non-standard packages
def bartlett_sphericity_test(dataset):
    corr = dataset.corr().values
    n = dataset.shape[0]
    p = dataset.shape[1]
    
    # Calculate the determinant of the correlation matrix
    det_corr = np.linalg.det(corr)
    
    # Calculate the test statistic
    statistic = -np.log(det_corr) * (n - 1 - (2 * p + 5) / 6)
    
    # Calculate degrees of freedom
    df = p * (p - 1) / 2
    
    # Calculate p-value
    p_value = 1.0 - stats.chi2.cdf(statistic, df)
    return statistic, p_value

chi_square_value, p_val = bartlett_sphericity_test(df_pharma)

print(f"Bartlett's Test Chi-Square Value: {chi_square_value:.2f}")
print(f"Bartlett's Test P-value: {p_val:.4e}")

if p_val < 0.05:
    print("\nConclusion: P-value < 0.05. We reject the null hypothesis. The dataset is suitable for Factor Analysis.")
else:
    print("\nConclusion: P-value >= 0.05. Variables are too independent. Factor Analysis may not be useful.")

## 5. Determining the Number of Factors: Eigenvalues and the Scree Plot
Before extracting factors, we must decide *how many* factors to retain. 

**The Kaiser Criterion:** Retain any factor with an Eigenvalue > 1. An eigenvalue represents the amount of variance explained by a factor. Since a single standardized variable has a variance of 1, a factor must explain at least as much variance as one single variable to be worth keeping.

**The Scree Plot:** A visual method where we plot eigenvalues and look for the 'elbow' or bend in the curve.

In [ ]:
# Standardize the data before calculating Eigenvalues
df_standardized = (df_pharma - df_pharma.mean()) / df_pharma.std()

# Calculate the correlation matrix and its eigenvalues
corr_matrix_std = df_standardized.corr()
eigenvalues, eigenvectors = np.linalg.eig(corr_matrix_std)

# Sort eigenvalues in descending order
eigenvalues = sorted(eigenvalues, reverse=True)

print("Eigenvalues for each potential factor:")
for i, ev in enumerate(eigenvalues):
    print(f"Factor {i+1}: {ev:.4f}")

# Plot the Scree Plot
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(eigenvalues) + 1), eigenvalues, marker='o', color='purple', linewidth=2)
plt.axhline(y=1, color='red', linestyle='--', label='Kaiser Criterion (Eigenvalue = 1)')
plt.title('Scree Plot for Factor Retention', fontsize=15)
plt.xlabel('Factor Number')
plt.ylabel('Eigenvalue (Variance Explained)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Decision: Only the first 3 factors have an Eigenvalue > 1. We will extract 3 factors.")

## 6. Core Concept 1: Factor Extraction (Unrotated)
Now we apply Factor Analysis to extract the 3 underlying factors. 

The output we care about most is the **Factor Loadings**. A factor loading is essentially the correlation coefficient between an observed variable and the latent factor. High loadings tell us which variables belong to which underlying factor.

In [ ]:
# Fit the Factor Analysis model (without rotation first to see the raw output)
fa_unrotated = FactorAnalysis(n_components=3, rotation=None, random_state=42)
fa_unrotated.fit(df_standardized)

# The components_ attribute contains the loadings (factors in rows, variables in columns)
# We transpose it for better readability
loadings_unrotated = pd.DataFrame(
    fa_unrotated.components_.T, 
    columns=['Factor_1', 'Factor_2', 'Factor_3'],
    index=df_standardized.columns
)

print("--- Unrotated Factor Loadings ---")
print(loadings_unrotated.round(3))

print("\nNotice that the unrotated loadings can be messy.")
print("Many variables might load moderately on multiple factors, making interpretation difficult.")

## 7. Core Concept 2: Rotation (The Secret Sauce)
Rotation is a mathematical technique used to make the factor structure more interpretable.
It aims for Simple Structure: every variable should have a high loading on exactly ONE factor, and near-zero loadings on all others.

**Varimax Rotation:** An orthogonal rotation method that keeps factors uncorrelated while maximizing the variance of squared loadings. This is the industry standard for business analytics.

In [ ]:
# Fit the Factor Analysis model with Varimax Rotation
fa_rotated = FactorAnalysis(n_components=3, rotation='varimax', random_state=42)
fa_rotated.fit(df_standardized)

loadings_rotated = pd.DataFrame(
    fa_rotated.components_.T, 
    columns=['Factor_1', 'Factor_2', 'Factor_3'],
    index=df_standardized.columns
)

print("--- Rotated Factor Loadings (Varimax) ---")
print(loadings_rotated.round(3))

## 8. Interpreting the Rotated Factors (Heatmap)
To present this to senior leadership, we visualize the rotated loadings using a heatmap. We look for loadings > 0.5 to declare that a variable strongly belongs to a factor.

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(loadings_rotated, annot=True, cmap='viridis', fmt=".2f", vmin=-1, vmax=1)
plt.title('Rotated Factor Loadings: Clear Business Structure', fontsize=15)
plt.ylabel('Observed KPIs')
plt.xlabel('Latent Factors')
plt.tight_layout()
plt.show()

print("Business Interpretation:")
print("- Factor_1 is heavily loaded with Delivery_Speed, Clinic_Density, and Supply_Efficiency.")
print("  -> We label this 'Product Accessibility'.")
print("- Factor_2 is heavily loaded with BMI, BP_Issues, and Cholesterol.")
print("  -> We label this 'Patient Health Profile'.")
print("- Factor_3 is heavily loaded with Brand_Trust, Sales_Growth, and Market_Share.")
print("  -> We label this 'Market Sentiment'.")

## 9. Common Variance vs. Unique Variance (Communalities)
In Factor Analysis, variance is split into two parts:
1. **Common Variance (Communality):** The proportion of a variable's variance explained by the extracted factors.
2. **Unique Variance:** The variance specific to that single variable alone (plus random error).

Communality is calculated by squaring the factor loadings and summing them across the factors for each variable.

In [ ]:
# Calculate Communalities (sum of squared loadings for each row)
communalities = np.sum(loadings_rotated**2, axis=1)

# Calculate Unique Variance (1 - Communality)
unique_variance = 1 - communalities

variance_df = pd.DataFrame({
    'Communality (Shared)': communalities,
    'Unique Variance (Noise)': unique_variance
})

print("--- Variance Breakdown per Variable ---")
print(variance_df.round(3))

print("\nInsight: Brand_Trust has a high communality (e.g., ~0.95), meaning the extracted factors explain 95% of its behavior.")
print("If a variable had a communality < 0.3, it would mean it has little in common with the rest of the dataset.")

## 10. Generating Factor Scores
The ultimate goal of data reduction is to replace the 9 original variables with the 3 new, consolidated factors.
We can transform our original dataset into **Factor Scores**. These scores represent each region's performance on the new 'super-variables'.

In [ ]:
# Transform the standardized data into Factor Scores
factor_scores = fa_rotated.transform(df_standardized)

# Create a new consolidated DataFrame
df_consolidated = pd.DataFrame(
    factor_scores, 
    columns=['Accessibility_Index', 'Health_Profile_Index', 'Sentiment_Index']
)

print("Original Dataset Shape:", df_pharma.shape)
print("Consolidated Dataset Shape:", df_consolidated.shape)
print("\n--- First 5 Regions on the new Super-Variables ---")
print(df_consolidated.head().round(3))

print("\nSuccess! We have reduced 9 complex KPIs down to 3 highly interpretable strategic drivers.")

## 11. Comparison: Regression vs. Factor Analysis
It is crucial to understand when to use which tool.

| Feature | Regression | Factor Analysis |
|---|---|---|
| **Primary Goal** | Prediction / Forecasting | Data Reduction / Structure Discovery |
| **Relationship** | Dependent variable (Y) vs. Predictors (X) | Inter-dependencies among all variables |
| **Output** | Coefficients / Weights | Factor Loadings / Factor Scores |
| **Business Use** | Estimating future sales | Consolidating 50 KPIs into 5 strategy drivers |

In [ ]:
# A simple demonstration comparing the outputs conceptually

print("REGRESSION THINKS:")
print("Y (Sales_Growth) = b1*(Brand_Trust) + b2*(Market_Share) + Error")
print("Goal: Find b1 and b2 to minimize Error in predicting Y.\n")

print("FACTOR ANALYSIS THINKS:")
print("Sales_Growth = L1*(Market_Sentiment_Factor) + Unique_Variance_1")
print("Brand_Trust  = L2*(Market_Sentiment_Factor) + Unique_Variance_2")
print("Market_Share = L3*(Market_Sentiment_Factor) + Unique_Variance_3")
print("Goal: Discover the hidden 'Market_Sentiment_Factor' that drives all three simultaneously.")

## 12. Practice Exercise 1: Exploring Customer Survey Data
**Scenario:** You have run a patient feedback survey with 6 questions, scored 1-10. 
Q1-Q3 ask about "Doctor Empathy, Listening Skills, Friendly Staff". 
Q4-Q6 ask about "Wait Time, Clinic Cleanliness, Billing Ease".

We will generate this synthetic survey dataset for you to analyze.

In [ ]:
# Generate Survey Data
n_surveys = 300
factor_interpersonal = np.random.normal(7, 1.5, n_surveys)
factor_operational = np.random.normal(6, 2.0, n_surveys)

df_survey = pd.DataFrame({
    'Q1_Empathy': factor_interpersonal + np.random.normal(0, 0.5, n_surveys),
    'Q2_Listening': factor_interpersonal + np.random.normal(0, 0.6, n_surveys),
    'Q3_Friendly': factor_interpersonal + np.random.normal(0, 0.4, n_surveys),
    'Q4_WaitTime': factor_operational + np.random.normal(0, 0.7, n_surveys),
    'Q5_Cleanliness': factor_operational + np.random.normal(0, 0.5, n_surveys),
    'Q6_Billing': factor_operational + np.random.normal(0, 0.8, n_surveys)
})

# Clip to 1-10 scale
df_survey = df_survey.clip(1, 10)
print("Survey Dataset Ready.")
print(df_survey.head())

### Exercise 1 Task:
Standardize the `df_survey` data. Calculate the correlation matrix and plot the Scree Plot to determine how many factors exist in this survey data.

In [ ]:
# --- EXERCISE 1 SOLUTION ---
# 1. Standardize
survey_std = (df_survey - df_survey.mean()) / df_survey.std()

# 2. Correlation Matrix and Eigenvalues
corr_survey = survey_std.corr()
evals, evecs = np.linalg.eig(corr_survey)
evals = sorted(evals, reverse=True)

# 3. Scree Plot
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(evals) + 1), evals, marker='s', color='darkblue')
plt.axhline(y=1, color='red', linestyle='--')
plt.title('Scree Plot for Survey Data')
plt.xlabel('Factor Number')
plt.ylabel('Eigenvalue')
plt.grid(alpha=0.3)
plt.show()

print("Based on the Kaiser Criterion (Eigenvalue > 1), we should extract 2 Factors.")

## 13. Practice Exercise 2: Factor Interpretation
### Exercise 2 Task:
Perform a Factor Analysis on the standardized survey data extracting 2 components with 'varimax' rotation. Plot the heatmap of the loadings and assign a business name to Factor 1 and Factor 2.

In [ ]:
# --- EXERCISE 2 SOLUTION ---
fa_survey = FactorAnalysis(n_components=2, rotation='varimax', random_state=42)
fa_survey.fit(survey_std)

survey_loadings = pd.DataFrame(
    fa_survey.components_.T,
    columns=['Factor_1', 'Factor_2'],
    index=survey_std.columns
)

plt.figure(figsize=(6, 5))
sns.heatmap(survey_loadings, annot=True, cmap='mako', fmt=".2f", vmin=-1, vmax=1)
plt.title('Survey Factor Loadings')
plt.tight_layout()
plt.show()

print("Business Interpretation:")
print("Factor 1 strongly represents Q4, Q5, Q6. We will name it 'Operational Excellence'.")
print("Factor 2 strongly represents Q1, Q2, Q3. We will name it 'Interpersonal Care'.")

## 14. Visualization Gallery: Unrotated vs. Rotated Loadings
To fully appreciate the "secret sauce" of Varimax rotation, let's plot the loadings on a 2D scatter plot before and after rotation. 
Rotation visually rotates the axes so that the points align more closely with a single axis (achieving Simple Structure).

In [ ]:
# Extract Unrotated and Rotated Loadings for the first 2 factors of our main Pharma dataset
loadings_unrot = fa_unrotated.components_.T[:, :2]
loadings_rot = fa_rotated.components_.T[:, :2]
variables = df_pharma.columns

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot Unrotated
axes[0].scatter(loadings_unrot[:, 0], loadings_unrot[:, 1], color='red', s=100)
for i, txt in enumerate(variables):
    axes[0].annotate(txt, (loadings_unrot[i, 0]+0.02, loadings_unrot[i, 1]+0.02), fontsize=9)
axes[0].set_title('Unrotated Loadings (Messy, mixed variables)')
axes[0].set_xlabel('Factor 1')
axes[0].set_ylabel('Factor 2')
axes[0].axhline(0, color='black', linewidth=1, linestyle='--')
axes[0].axvline(0, color='black', linewidth=1, linestyle='--')

# Plot Rotated
axes[1].scatter(loadings_rot[:, 0], loadings_rot[:, 1], color='blue', s=100)
for i, txt in enumerate(variables):
    axes[1].annotate(txt, (loadings_rot[i, 0]+0.02, loadings_rot[i, 1]+0.02), fontsize=9)
axes[1].set_title('Rotated Loadings (Varimax) (Aligned to Axes)')
axes[1].set_xlabel('Factor 1')
axes[1].set_ylabel('Factor 2')
axes[1].axhline(0, color='black', linewidth=1, linestyle='--')
axes[1].axvline(0, color='black', linewidth=1, linestyle='--')

plt.tight_layout()
plt.show()

print("Visual Insight: Rotation literally spins the axis grid to snap as many points to the x or y axis as possible.")
print("This pushes loadings toward 1.0 or 0.0, eliminating ambiguity.")

## 15. Summary and Key Takeaways

- **Purpose:** Factor Analysis is for Data Reduction and Latent Structure Discovery. It simplifies high-dimensional data into a few meaningful, unobservable drivers.
- **Assumptions:** Your data must have significant correlations. We test this using Bartlett's Test of Sphericity and the KMO measure.
- **Factor Retention:** Use the Kaiser Criterion (Eigenvalues > 1) and the visual Scree Plot to determine the number of latent factors.
- **Rotation is Essential:** Unrotated factors are mathematically valid but practically uninterpretable. Varimax rotation forces a Simple Structure, making business interpretation clear.
- **Output & Application:** Factor Loadings tell you the relationship between variables and factors. Factor Scores allow you to replace your massive dataset with a few consolidated super-variables for future modeling.

In [ ]:
print("Module 9: Factor Analysis completed successfully!")
print("You are now equipped to handle high-dimensional business data and uncover the hidden architecture within it.")